[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_25/2_sql.ipynb)

# Day 25: SQL (hard)

**Company style: TikTok.** Short-video data: watch time, completion, sessions, follows, activation. TikTok SQL rounds are usually 2 to 3 queries in 30 to 45 minutes with follow-up questions on the metric definition, so say your definition out loud before you write the query.

**How to work:** Read the tables first (columns, one row = what?). Write the query in your head in the order FROM, WHERE, GROUP BY, HAVING, SELECT, ORDER BY. Check the result for duplicates and NULLs before you open the solution.

**Today (32 min):**

| # | Question | Level | Minutes |
|---|---|---|---|
| Q1 | Activation within 24 hours | hard | 6 |
| Q2 | Most watched-through creators per category | hard | 6 |
| Q3 | Viewing sessions | hard | 8 |
| Q4 | Do new viewers come back? (review) | hard | 6 |
| Q5 | Creators who turn viewers into followers (review) | hard | 6 |

**Handbook:** [SQL](https://mahsa-agz.github.io/ds-handbook/#sql), [cheat sheets](https://mahsa-agz.github.io/ds-handbook/#cheat)

## Tables

**Made-up short-video tables** (invented for practice, fixed seed; May 2024):

| Table | One row is | Columns |
|---|---|---|
| `tt_creators` | one creator (40) | `creator_id`, `category` (main category), `country` |
| `tt_videos` | one posted video | `video_id`, `creator_id`, `posted_at` ('YYYY-MM-DD HH:MM:SS'), `duration_s`, `category` |
| `tt_views` | one play of a video | `view_id`, `viewer_id`, `video_id`, `view_time`, `watch_s` (seconds watched; can be MORE than `duration_s` when the video loops), `source` (fyp / following) |
| `tt_follows` | one follow | `follower_id`, `creator_id`, `followed_at` |

**Made-up signup tables (tiny):** `tt_signups` (`user_id`, `signup_time`, `country`): one row per new account.
`tt_texts` (`text_id`, `user_id`, `sent_at`, `status` = confirmed / not_confirmed): one row per verification SMS; a
user can get several texts, and `sent_at` of a confirmed text is the moment the user confirmed.

The setup cell prints every table with its columns. Call `tables()` again any time.

## Setup

Run this cell once before the questions.

In [ ]:
import sqlite3
import pandas as pd

def run(q, con=None):
    """Runs a query and shows the result as a table (SELECT / WITH); other statements just run."""
    con = con or db
    if not q.strip():
        print("Write your query between the triple quotes, then run the cell.")
        return None
    if q.strip().lower().startswith(("select", "with")):
        return pd.read_sql_query(q, con)
    con.executescript(q)
    print("done")

# Finds the data folder (local copy of the repository, or /content/data in Colab) and downloads a missing file
# from the ds-handbook repository on GitHub (or from its original source).
import os, sys, urllib.request

RAW = "https://raw.githubusercontent.com/mahsa-agz/mahsa-agz.github.io/main/ds-handbook/exam_prep/data/"

def _find_data():
    if "google.colab" in sys.modules:
        os.makedirs("/content/data", exist_ok=True)
        return "/content/data"
    for p in ("../data", "data", "exam_prep/data"):
        if os.path.isdir(p):
            return p
    os.makedirs("data", exist_ok=True)
    return "data"

DATA = _find_data()
print("data folder:", DATA)

def data_path(name, url=None):
    """Local path of a data file; downloads it from the repository (or from url) if it is not there yet."""
    path = os.path.join(DATA, name)
    if not os.path.exists(path):
        for source in (RAW + name, url):
            if not source:
                continue
            try:
                print("downloading", name)
                urllib.request.urlretrieve(source, path)
                break
            except Exception as e:
                print("  could not download from", source, "->", e)
    return path

db = sqlite3.connect(':memory:')

# Made-up short-video tables (generated with a fixed seed; numpy's legacy RandomState is frozen across versions).
import numpy as np

def make_video_tables(con, seed=25):
    rs = np.random.RandomState(seed)
    cats = ["dance", "food", "comedy", "education", "gaming"]
    nc = 40
    creators = pd.DataFrame({
        "creator_id": np.arange(1, nc + 1),
        "category": [cats[i] for i in rs.randint(0, 5, nc)],
        "country": [["US", "KR", "BR", "ID", "DE"][i] for i in rs.choice(5, nc, p=[.3, .25, .2, .15, .1])],
    })
    pop = rs.lognormal(0, 1.0, nc)                              # creator popularity
    start = pd.Timestamp("2024-05-01")
    vids = []
    vid = 1000
    for c in range(nc):
        for _ in range(1 + rs.poisson(6)):
            vid += 1
            posted = start + pd.Timedelta(seconds=int(rs.randint(0, 27 * 86400)))
            cat = creators.category[c] if rs.rand() < 0.85 else cats[rs.randint(0, 5)]
            vids.append((vid, c + 1, posted, int(rs.choice([15, 30, 45, 60, 90, 180])), cat))
    videos = pd.DataFrame(vids, columns=["video_id", "creator_id", "posted_at", "duration_s", "category"])
    videos = videos.sort_values("posted_at", kind="mergesort").reset_index(drop=True)
    vq = rs.beta(2, 2, len(videos))                             # video quality: how much of it people watch
    w = pop[videos.creator_id.values - 1] * (0.5 + vq)
    posted_s = ((videos.posted_at - start).dt.total_seconds()).values
    views, follows, followed = [], [], set()
    for u in range(1, 1201):
        n_sess = rs.poisson(4)
        for _ in range(n_sess):
            t = int(rs.randint(1 * 86400, 31 * 86400))
            n_vid = 1 + rs.geometric(1 / 6.0)
            source = "following" if rs.rand() < 0.15 else "fyp"
            for _ in range(n_vid):
                ok = np.nonzero(posted_s < t)[0]
                if len(ok) == 0:
                    break
                p = w[ok] / w[ok].sum()
                i = ok[rs.choice(len(ok), p=p)]
                dur = videos.duration_s[i]
                frac = min(rs.beta(1 + 4 * vq[i], 2), 1.0)
                if rs.rand() < 0.08:
                    frac += 1 + rs.rand()                       # rewatch loop: watched more than the length
                watch = max(1, int(round(frac * dur)))
                views.append((u, int(videos.video_id[i]), start + pd.Timedelta(seconds=t), watch, source))
                cid = int(videos.creator_id[i])
                if frac > 0.8 and (u, cid) not in followed and rs.rand() < 0.06:
                    followed.add((u, cid))
                    follows.append((u, cid, start + pd.Timedelta(seconds=t + watch + 2)))
                t += watch + int(rs.randint(1, 15))
                source = "fyp"
    fmt = "%Y-%m-%d %H:%M:%S"
    views = pd.DataFrame(views, columns=["viewer_id", "video_id", "view_time", "watch_s", "source"])
    views = views.sort_values(["view_time", "viewer_id"], kind="mergesort").reset_index(drop=True)
    views.insert(0, "view_id", np.arange(1, len(views) + 1))
    views["view_time"] = views["view_time"].dt.strftime(fmt)
    follows = pd.DataFrame(follows, columns=["follower_id", "creator_id", "followed_at"])
    follows = follows.sort_values("followed_at", kind="mergesort").reset_index(drop=True)
    follows["followed_at"] = follows["followed_at"].dt.strftime(fmt)
    videos["posted_at"] = videos["posted_at"].dt.strftime(fmt)
    for name, df in [("tt_creators", creators), ("tt_videos", videos), ("tt_views", views), ("tt_follows", follows)]:
        df.to_sql(name, con, index=False, if_exists="replace")

make_video_tables(db)

# Made-up signup verification tables (tiny on purpose: check the answer by hand).
db.executescript("""
CREATE TABLE tt_signups (user_id INTEGER, signup_time TEXT, country TEXT);
INSERT INTO tt_signups VALUES
(1, '2024-05-01 10:00:00', 'US'), (2, '2024-05-01 11:30:00', 'US'), (3, '2024-05-01 23:50:00', 'KR'),
(4, '2024-05-02 08:00:00', 'KR'), (5, '2024-05-02 09:15:00', 'BR'), (6, '2024-05-02 20:00:00', 'US'),
(7, '2024-05-03 07:45:00', 'KR'), (8, '2024-05-03 12:00:00', 'BR'), (9, '2024-05-03 18:30:00', 'US'),
(10, '2024-05-04 06:10:00', 'KR');
CREATE TABLE tt_texts (text_id INTEGER, user_id INTEGER, sent_at TEXT, status TEXT);
INSERT INTO tt_texts VALUES
(101, 1, '2024-05-01 10:00:05', 'confirmed'),
(102, 2, '2024-05-01 11:30:04', 'not_confirmed'), (103, 2, '2024-05-01 15:00:00', 'confirmed'),
(104, 3, '2024-05-01 23:50:03', 'not_confirmed'), (105, 3, '2024-05-02 23:00:00', 'confirmed'),
(106, 4, '2024-05-02 08:00:02', 'not_confirmed'), (107, 4, '2024-05-03 09:00:00', 'confirmed'),
(108, 5, '2024-05-02 09:15:03', 'confirmed'), (109, 5, '2024-05-02 09:20:00', 'confirmed'),
(110, 7, '2024-05-03 07:45:02', 'not_confirmed'), (115, 7, '2024-05-03 20:00:00', 'confirmed'),
(111, 8, '2024-05-03 12:00:03', 'confirmed'),
(112, 9, '2024-05-03 18:30:02', 'not_confirmed'), (113, 9, '2024-05-05 10:00:00', 'confirmed'),
(114, 10, '2024-05-04 06:10:01', 'confirmed');
""")

def tables():
    """Prints every table with its row count and columns."""
    for (t,) in db.execute("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name"):
        cols = [r[1] for r in db.execute(f'PRAGMA table_info("{t}")')]
        n = db.execute(f'SELECT COUNT(*) FROM "{t}"').fetchone()[0]
        print(f"{t} ({n:,} rows): {', '.join(cols)}")

tables()

---
### Q1. Activation within 24 hours

*hard, about 6 min*

A new account is **activated** if the user confirms a verification text within 24 hours of
`signup_time` (any of their texts counts). Return the activation rate per `country` and one extra row with country
'ALL' for the total: `country`, `signups`, `activated`, `pct` (1 decimal). Country rows first (highest pct first),
then 'ALL'.

Edge cases in the data: a user who got two texts and confirmed the second; a user who confirmed twice; a user who never
got a text; a user who confirmed 23 hours later but on the next calendar day.

*Follow-up:* product wants "confirmed on the day after signup" instead. What changes, and why is that metric worse?

*Source: [DataLemur TikTok SQL questions (inspiration, own wording and data)](https://datalemur.com/blog/tiktok-sql-interview-questions)*

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: an activation rate where users can have 0, 1 or many texts. Pattern: ratio with LEFT JOIN + COUNT(DISTINCT CASE ...), and UNION ALL for the total row.

</details>

<details><summary><b>Hint 2</b></summary>

1. `tt_signups s LEFT JOIN tt_texts t ON t.user_id = s.user_id`.
2. activated = `COUNT(DISTINCT CASE WHEN t.status = 'confirmed' AND t.sent_at <= datetime(s.signup_time, '+24 hours') THEN s.user_id END)`.
3. Put the per-country query and an 'ALL' query (no GROUP BY) in a CTE with UNION ALL; sort with a CASE.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH j AS (
  SELECT s.user_id, s.country,
    CASE WHEN t.status = 'confirmed'
          AND t.sent_at <= datetime(s.signup_time, '+24 hours') THEN s.user_id END AS ok_user
  FROM tt_signups s
  LEFT JOIN tt_texts t ON t.user_id = s.user_id
), rates AS (
  SELECT country, COUNT(DISTINCT user_id) AS signups, COUNT(DISTINCT ok_user) AS activated
  FROM j GROUP BY country
  UNION ALL
  SELECT 'ALL', COUNT(DISTINCT user_id), COUNT(DISTINCT ok_user) FROM j
)
SELECT country, signups, activated,
  ROUND(100.0 * activated / signups, 1) AS pct
FROM rates
ORDER BY CASE WHEN country = 'ALL' THEN 1 ELSE 0 END, pct DESC
```

Result:

| country | signups | activated | pct |
|---|---|---|---|
| BR | 2 | 2 | 100.0 |
| KR | 4 | 3 | 75.0 |
| US | 4 | 2 | 50.0 |
| ALL | 10 | 7 | 70.0 |

**Why:** The LEFT JOIN keeps user 6 (no text) in the denominator; COUNT(DISTINCT ...) counts user 5 (two confirmations) and user 2 (two texts) once; the 24 hour test on timestamps accepts user 3 (23 h 10 min, next calendar day) and rejects user 4 (25 h) and user 9 (about 40 h). Result: BR 100.0, KR 75.0, US 50.0, ALL 70.0. Follow-up: a calendar-day definition depends on the signup hour (a 23:50 signup has 10 minutes on day 0) and on the time zone, so it is unfair across users; a fixed 24 hour window is the same for everyone.

**Complexity:** One join and two small aggregations.

**Common mistakes:** Inner join (user 6 disappears: rate too high). Counting texts instead of users. `date(sent_at) = date(signup_time)` (rejects user 3). PostgreSQL: `t.sent_at <= s.signup_time + interval '24 hours'`.

**Learn more:** [sql-ratios](https://mahsa-agz.github.io/ds-handbook/#sql-ratios), [sql-left-join-nulls](https://mahsa-agz.github.io/ds-handbook/#sql-left-join-nulls), [sql-dates](https://mahsa-agz.github.io/ds-handbook/#sql-dates)

</details>

---
### Q2. Most watched-through creators per category

*hard, about 6 min*

Completion of one view = `watch_s / duration_s`, **capped at 1** (a looping video can be watched longer
than it lasts). A creator's completion rate is the average over all views of all their videos. Among creators with at
least 200 views, return the **top 2 per category** (the creator's category in `tt_creators`): `category`,
`creator_id`, `views`, `pct` (completion in %, 1 decimal), ordered by category then rank.

Example: a 30 s video watched 45 s (a loop) counts as 1.0, not 1.5.

*Follow-up:* why cap at 1 instead of using raw watch time?

*Source: [DataLemur TikTok SQL questions (inspiration, own wording and data)](https://datalemur.com/blog/tiktok-sql-interview-questions)*

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: average of a per-row ratio, then 'top 2 per group'. Pattern: ratios (cap with MIN(x, 1.0) or CASE) plus top-N per group with RANK / ROW_NUMBER.

</details>

<details><summary><b>Hint 2</b></summary>

1. Per view: `MIN(1.0 * watch_s / duration_s, 1.0)` (scalar MIN in SQLite; LEAST in PostgreSQL).
2. GROUP BY creator with HAVING COUNT(*) >= 200.
3. `RANK() OVER (PARTITION BY category ORDER BY avg DESC)`; keep <= 2.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH v AS (
  SELECT vd.creator_id,
    CASE WHEN w.watch_s >= vd.duration_s THEN 1.0
         ELSE 1.0 * w.watch_s / vd.duration_s END AS completion
  FROM tt_views w
  JOIN tt_videos vd ON vd.video_id = w.video_id
), c AS (
  SELECT cr.category, v.creator_id, COUNT(*) AS views, AVG(v.completion) AS avg_completion
  FROM v JOIN tt_creators cr ON cr.creator_id = v.creator_id
  GROUP BY cr.category, v.creator_id
  HAVING COUNT(*) >= 200
), r AS (
  SELECT *, RANK() OVER (PARTITION BY category ORDER BY avg_completion DESC) AS rk
  FROM c
)
SELECT category, creator_id, views, ROUND(100 * avg_completion, 1) AS pct
FROM r
WHERE rk <= 2
ORDER BY category, rk
```

Result:

| category | creator_id | views | pct |
|---|---|---|---|
| comedy | 25 | 1893 | 69.5 |
| comedy | 39 | 1255 | 64.3 |
| dance | 35 | 769 | 67.0 |
| dance | 17 | 619 | 64.4 |
| education | 13 | 231 | 69.0 |
| education | 15 | 524 | 65.8 |
| food | 10 | 1322 | 65.4 |
| food | 16 | 6574 | 63.8 |
| gaming | 33 | 342 | 66.5 |
| gaming | 9 | 1332 | 66.0 |

**Why:** Capping happens per view BEFORE averaging. The CASE form works in every dialect. The HAVING filter runs before ranking, so a creator with 20 perfect views cannot win. Comedy creator 25 leads overall with 69.5%. Follow-up: without the cap 8% of views (the loops) count up to 3 times; the average becomes 72.1% instead of 63.2%, and creators of short looping videos win by construction. Watch time is a separate metric (total attention); completion should measure 'did people finish it'.

**Complexity:** One join, one aggregation, one window over 40 creators.

**Common mistakes:** AVG(watch_s) / AVG(duration_s) (a ratio of averages, a different metric). Ranking before the 200 view filter. Integer division (`watch_s / duration_s` is 0 for most views).

**Learn more:** [sql-ratios](https://mahsa-agz.github.io/ds-handbook/#sql-ratios), [sql-top-n](https://mahsa-agz.github.io/ds-handbook/#sql-top-n)

</details>

---
### Q3. Viewing sessions

*hard, about 8 min*

A viewer's views belong to the same **session** until there is a break of more than 30 minutes between the
END of one view (`view_time + watch_s` seconds) and the START of the next. Return one row: `sessions`, `viewers`,
`per_viewer` (sessions per viewer), `avg_videos` (videos per session), `avg_min` (session length in minutes, from
first start to last end), all with 2 decimals (avg_min with 1), plus `max_videos`.

Example: views at 20:00 (60 s), 20:05 (30 s), 21:10 (10 s): the gap 20:05:30 to 21:10 is 64.5 minutes, so 2 sessions.

*Follow-up:* the events arrive as a stream. How would the job that builds sessions work?

*Source: [DataLemur TikTok SQL questions (inspiration, own wording and data)](https://datalemur.com/blog/tiktok-sql-interview-questions)*

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: 'a break of more than N minutes starts a new one'. Pattern: sessionization = gaps and islands with LAG (flag a new session) plus a running SUM of the flags (session number).

</details>

<details><summary><b>Hint 2</b></summary>

1. `LAG(datetime(view_time, '+' || watch_s || ' seconds'))` per viewer = end of the previous view.
2. new_session = 1 if no previous view or the gap > 30 minutes.
3. `SUM(new_session) OVER (PARTITION BY viewer_id ORDER BY view_time ROWS UNBOUNDED PRECEDING)` = session id.
4. GROUP BY viewer, session; then one summary row.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH v AS (
  SELECT viewer_id, view_id, view_time, watch_s,
    datetime(view_time, '+' || watch_s || ' seconds') AS end_time,
    LAG(datetime(view_time, '+' || watch_s || ' seconds'))
      OVER (PARTITION BY viewer_id ORDER BY view_time, view_id) AS prev_end
  FROM tt_views
), flagged AS (
  SELECT *,
    CASE WHEN prev_end IS NULL
           OR (julianday(view_time) - julianday(prev_end)) * 1440 > 30 THEN 1 ELSE 0 END AS new_session
  FROM v
), numbered AS (
  SELECT *,
    SUM(new_session) OVER (PARTITION BY viewer_id ORDER BY view_time, view_id
                           ROWS UNBOUNDED PRECEDING) AS session_id
  FROM flagged
), sessions AS (
  SELECT viewer_id, session_id, COUNT(*) AS videos,
    (julianday(MAX(end_time)) - julianday(MIN(view_time))) * 1440 AS minutes
  FROM numbered
  GROUP BY viewer_id, session_id
)
SELECT COUNT(*) AS sessions,
  COUNT(DISTINCT viewer_id) AS viewers,
  ROUND(1.0 * COUNT(*) / COUNT(DISTINCT viewer_id), 2) AS per_viewer,
  ROUND(AVG(videos), 2) AS avg_videos,
  ROUND(AVG(minutes), 1) AS avg_min,
  MAX(videos) AS max_videos
FROM sessions
```

Result:

| sessions | viewers | per_viewer | avg_videos | avg_min | max_videos |
|---|---|---|---|---|---|
| 4843 | 1183 | 4.09 | 6.99 | 6.3 | 57 |

**Why:** The gap is measured from the END of the previous view, otherwise one long video would split a session. The flag-then-running-sum trick turns 'new session starts here' into a session number that GROUP BY can use. 4,843 sessions from 1,183 viewers, about 7 videos and 6.3 minutes each. Follow-up: in streaming, keep per viewer state (last end time, current session id); close a session when 30 minutes pass with no event (a timer / watermark), and handle late events by re-opening within an allowed lateness.

**Complexity:** One sort per viewer for LAG and the running sum: O(n log n).

**Common mistakes:** Gap from the previous START. `ORDER BY view_time` alone when two views share a timestamp (add view_id for a stable order). Default window frame RANGE in the running sum (ties get the same total). PostgreSQL: `view_time + watch_s * interval '1 second'`.

**Learn more:** [sql-gaps-islands](https://mahsa-agz.github.io/ds-handbook/#sql-gaps-islands), [sql-running-totals](https://mahsa-agz.github.io/ds-handbook/#sql-running-totals), [sql-lag-lead](https://mahsa-agz.github.io/ds-handbook/#sql-lag-lead)

</details>

---
### Q4. Do new viewers come back? (review)

*hard, about 6 min*

A viewer's day 0 is the date of their first view. Group viewers by the **week** (Monday start) of day 0.
For each cohort week return `viewers`, `d1_pct` (viewed on day 1) and `w1_pct` (viewed on ANY day from day 1 to day 7),
1 decimal. Data ends on 2024-05-31, so keep only viewers whose day 7 is inside the data (day 0 on or before 2024-05-24).

*Follow-up:* `w1_pct` is about 5 times `d1_pct`. Which one would you put in a weekly business review, and why?

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: 'cohort', 'came back on day N / within N days'. Pattern: retention: first date per user, join activity, MAX(CASE ...) per user as 0/1 flags, then AVG per cohort.

</details>

<details><summary><b>Hint 2</b></summary>

1. `first`: MIN(date(view_time)) per viewer.
2. Join all views; per viewer: `MAX(CASE WHEN date(view_time) = date(d0, '+1 day') THEN 1 ELSE 0 END)` and the same with BETWEEN +1 and +7 days.
3. Filter `d0 <= '2024-05-24'`; group by `date(d0, 'weekday 0', '-6 days')`.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH first AS (
  SELECT viewer_id, MIN(date(view_time)) AS d0
  FROM tt_views GROUP BY viewer_id
), flags AS (
  SELECT f.viewer_id, f.d0,
    date(f.d0, 'weekday 0', '-6 days') AS cohort_week,
    MAX(CASE WHEN date(v.view_time) = date(f.d0, '+1 day') THEN 1 ELSE 0 END) AS d1,
    MAX(CASE WHEN date(v.view_time) BETWEEN date(f.d0, '+1 day') AND date(f.d0, '+7 days')
             THEN 1 ELSE 0 END) AS w1
  FROM first f
  JOIN tt_views v ON v.viewer_id = f.viewer_id
  WHERE f.d0 <= '2024-05-24'
  GROUP BY f.viewer_id, f.d0
)
SELECT cohort_week, COUNT(*) AS viewers,
  ROUND(100.0 * AVG(d1), 1) AS d1_pct,
  ROUND(100.0 * AVG(w1), 1) AS w1_pct
FROM flags
GROUP BY cohort_week
ORDER BY cohort_week
```

Result:

| cohort_week | viewers | d1_pct | w1_pct |
|---|---|---|---|
| 2024-04-29 | 486 | 12.8 | 63.0 |
| 2024-05-06 | 412 | 11.2 | 59.5 |
| 2024-05-13 | 195 | 16.4 | 65.6 |
| 2024-05-20 | 50 | 8.0 | 64.0 |

**Why:** One row per viewer with 0/1 flags makes AVG a rate and avoids double counting viewers with many views. The observability filter removes viewers whose 7 day window is cut by the end of the data (they would look churned). The first cohort is the week of 2024-04-29 (May 1 is a Wednesday). Follow-up: w1 (about 60% to 66%) is stable and less sensitive to the weekday of day 0, so it is better for a weekly review; d1 (8% to 16%) is noisy but reacts faster, so it is a good early warning in experiments.

**Complexity:** One join of views to first dates and one GROUP BY per viewer.

**Common mistakes:** Counting day 0 as a return. Keeping recent cohorts with incomplete windows. COUNT(*) after the join (counts views, not viewers).

**Learn more:** [sql-retention](https://mahsa-agz.github.io/ds-handbook/#sql-retention), [sql-dates](https://mahsa-agz.github.io/ds-handbook/#sql-dates)

</details>

---
### Q5. Creators who turn viewers into followers (review)

*hard, about 6 min*

Follow rate of a creator = follows received / **distinct** viewers of their videos. For creators with at
least 100 distinct viewers, return `category`, `creator_id`, `viewers`, `follows`, `pct` (follow rate in %, 2
decimals), `cat_avg` (the average follow rate of qualifying creators in the same category, %, 2 decimals) and `idx` =
creator rate / category average (2 decimals). Show the top 5 by `idx`.

*Follow-up:* why compare with the category average instead of ranking raw follow rates?

*Source: [DataLemur TikTok SQL questions (inspiration, own wording and data)](https://datalemur.com/blog/tiktok-sql-interview-questions)*

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: a rate per creator compared with its group. Pattern: ratios, pre-aggregated per creator, then a window AVG OVER (PARTITION BY category).

</details>

<details><summary><b>Hint 2</b></summary>

1. `uv`: COUNT(DISTINCT viewer_id) per creator (views join videos).
2. `f`: follows per creator.
3. LEFT JOIN f (creators with 0 follows), HAVING / WHERE viewers >= 100.
4. `AVG(rate) OVER (PARTITION BY category)` and rate / that.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH uv AS (
  SELECT vd.creator_id, COUNT(DISTINCT w.viewer_id) AS viewers
  FROM tt_views w JOIN tt_videos vd ON vd.video_id = w.video_id
  GROUP BY vd.creator_id
), f AS (
  SELECT creator_id, COUNT(*) AS follows FROM tt_follows GROUP BY creator_id
), x AS (
  SELECT c.category, uv.creator_id, uv.viewers, COALESCE(f.follows, 0) AS follows,
    1.0 * COALESCE(f.follows, 0) / uv.viewers AS rate
  FROM uv
  JOIN tt_creators c ON c.creator_id = uv.creator_id
  LEFT JOIN f ON f.creator_id = uv.creator_id
  WHERE uv.viewers >= 100
)
SELECT category, creator_id, viewers, follows,
  ROUND(100 * rate, 2) AS pct,
  ROUND(100 * AVG(rate) OVER (PARTITION BY category), 2) AS cat_avg,
  ROUND(rate / AVG(rate) OVER (PARTITION BY category), 2) AS idx
FROM x
ORDER BY idx DESC
LIMIT 5
```

Result:

| category | creator_id | viewers | follows | pct | cat_avg | idx |
|---|---|---|---|---|---|---|
| food | 16 | 1118 | 89 | 7.96 | 3.1 | 2.57 |
| comedy | 25 | 815 | 40 | 4.91 | 2.34 | 2.1 |
| education | 15 | 392 | 12 | 3.06 | 2.03 | 1.51 |
| dance | 35 | 505 | 16 | 3.17 | 2.19 | 1.45 |
| gaming | 22 | 653 | 24 | 3.68 | 2.58 | 1.42 |

**Why:** Views and follows are aggregated separately and then joined (joining raw views to raw follows would multiply rows). The window AVG keeps every creator row while adding the category benchmark. Food creator 16 converts 7.96% of viewers, 2.57 times the food average. Follow-up: categories have different natural follow rates (food 3.10% versus education 2.03% on average), so raw ranking mostly ranks categories; the index finds creators who beat their peers.

**Complexity:** Two aggregations and one window over about 40 rows.

**Common mistakes:** Denominator = views (a viewer who watches 20 videos counts 20 times). Joining views to follows before aggregating. Average of the category computed before the 100 viewer filter (a different benchmark; say which you use).

**Learn more:** [sql-ratios](https://mahsa-agz.github.io/ds-handbook/#sql-ratios), [sql-window-ranking](https://mahsa-agz.github.io/ds-handbook/#sql-window-ranking)

</details>

---
## Redo list

Write down every question you got wrong, solved only after Hint 2, or finished over time. Solve it again from scratch 3 days and 7 days later.

**Next:** [Statistics, A/B testing and product](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_25/3_stats.ipynb)